# [투자전략] 같은 미국 ETF인데 세금이 다르다? 1부 - 계좌별 세금 차이와 절세 원리

아래 코드 셀을 실행하면 계좌별 세금 표와 보유기간별 세후 인출액 표를 표시합니다.


In [ ]:
# @title 투자 조건을 입력하고 계좌별 세후 금액을 비교하세요
초기투자금_억원 = 1.0  # @param {type:"number", min:0.01, max:1.0, step:0.1}
연수익률_퍼센트 = 10.0  # @param {type:"number", min:0.1, step:0.1}
최대인출시점_년 = 10  # @param {type:"integer", min:1, max:10, step:1}
국내일반계좌_매도방식 = "이익 2,000만원 한도 내 매도·재매수"  # @param ["매년 전량 매도·재매수", "이익 2,000만원 한도 내 매도·재매수"]

"""미국 상장 ETF 투자 시 계좌별 세금과 보유기간별 세후 인출액을 시뮬레이션으로 비교한다."""

WON_PER_MANWON = 10_000
WON_PER_EOK = 100_000_000
RETURN_RATE = float(연수익률_퍼센트) / 100
INITIAL_INVESTMENT_KRW = float(초기투자금_억원) * WON_PER_EOK
MAX_WITHDRAWAL_YEARS = int(최대인출시점_년)
US_DEDUCTION_KRW = 2_500_000
US_TAX_RATE = 0.22
US_EFFECTIVE_RATE_EXEMPT_PRINCIPAL_KRW = 25_000_000
GENERAL_TAX_RATE = 0.154
FINANCIAL_INCOME_LIMIT_KRW = 20_000_000
GENERAL_MAX_KRW = FINANCIAL_INCOME_LIMIT_KRW / RETURN_RATE
ISA_EXEMPTION_KRW = 2_000_000
ISA_TAX_RATE = 0.099
ISA_MAX_KRW = 100_000_000
PENSION_TAX_RATE = 0.165
GENERAL_ANNUAL_SALE_MODE = "매년 전량 매도·재매수"
GENERAL_THRESHOLD_SALE_MODE = "이익 2,000만원 한도 내 매도·재매수"
GENERAL_SALE_MODE = str(국내일반계좌_매도방식)

ASSUMPTIONS_MD = f"""
## 계산 가정

- 미국계좌에서는 미국 상장 ETF에, 국내 일반계좌·ISA·연금저축에서는 동일한 수익률을 가정한 국내상장 해외 ETF에 투자합니다.
- **1년간 투자**한 뒤 전액 매도하고, 해당 계좌의 과세 방식에 따라 세금을 계산합니다.
- 연 수익률은 **{RETURN_RATE:.0%}**이며, 분배금 없이 `투자금액 × {RETURN_RATE:.0%}` 전부를 과세대상 투자수익으로 봅니다. 비교를 단순화하기 위해 국내상장 해외 ETF도 투자수익 전부가 과세 대상이라고 가정하며, 실제 과표기준가격과의 차이는 반영하지 않습니다.
- 미국계좌는 양도차익에서 연 250만원을 공제한 뒤 {US_TAX_RATE:.1%}를 과세합니다. 투자금액별 실효세율 비교에서는 투자금 2,500만원까지 0%, 초과 투자금의 수익에 {US_TAX_RATE:.1%}를 적용합니다.
- 국내 일반계좌는 투자수익에 {GENERAL_TAX_RATE:.1%}를 과세합니다. 다른 이자·배당소득은 없다고 가정하며, 투자수익이 2,000만원을 초과하면 금융소득종합과세 대상이 될 수 있으므로 최저세율 계좌 후보에서 제외합니다.
- ISA 납입금액은 최대 1억원으로 제한합니다. 투자금액별 실효세율 비교에서는 비과세 혜택을 제외하고 투자수익 전액에 {ISA_TAX_RATE:.1%}를 적용합니다.
- 연금저축 원금은 세액공제를 받지 않은 것으로 가정합니다. 1년 뒤 연금 외 수령하는 운용수익에 {PENSION_TAX_RATE:.1%}를 과세합니다.
- 투자금액별 실효세율 비교에서는 계좌 자체의 세율만 비교하기 위해 연금저축 납입한도를 적용하지 않습니다. 실제 절세 계좌 배분 전략에는 연금계좌 합산 연 1,800만원 한도를 적용합니다.
- 다른 이자·배당소득, 매매·환전 수수료, 해외 원천징수와 추적오차는 반영하지 않습니다.
- 총 투자금 전액을 한 계좌에 투자했을 때의 실효세율을 비교합니다. 최저세율이 같은 계좌는 모두 표시합니다.
"""
WITHDRAWAL_ASSUMPTIONS_MD = f"""
## 보유기간별 세후 인출액 계산 가정

- 초기 투자금 **{초기투자금_억원:g}억원**을 연 **{RETURN_RATE:.1%}**로 복리 운용하고, 1년부터 {MAX_WITHDRAWAL_YEARS}년까지 각 종료 시점에 전액 매도·인출한 독립 시나리오를 비교합니다.
- 미국계좌의 연 250만원 양도소득 기본공제는 이미 사용한 것으로 가정하고, 종료연도의 누적 양도차익 전체에 {US_TAX_RATE:.1%}를 과세합니다.
- 국내 일반계좌 매도 방식은 **{GENERAL_SALE_MODE}**입니다. 매년 전량 매도·재매수 방식은 발생이익 전체에, 이익 2,000만원 한도 내 매도·재매수 방식은 미실현이익이 2,000만원을 초과하는 해에 실현한 이익 2,000만원에 {GENERAL_TAX_RATE:.1%}를 과세합니다. 후자의 표에는 남은 미실현이익도 {GENERAL_TAX_RATE:.1%}로 시뮬레이션 청산한 금액을 표시합니다.
- ISA는 중간 매도 없이 종료연도에 전액 매도하며, 누적수익 200만원 비과세 후 초과분에 {ISA_TAX_RATE:.1%}를 과세합니다.
- 연금저축은 중간 매도 없이 종료연도에 전액 매도·인출하며, 세액공제를 받지 않은 원금을 제외한 누적수익에 {PENSION_TAX_RATE:.1%}를 과세합니다.
- 분배금, 수수료, 환율, 원천징수와 상품별 추적오차는 반영하지 않습니다. 세금은 모두 계좌자산에서 차감합니다.
"""


def format_korean_currency(amount_krw: float) -> str:
    """원화 금액을 억원·만원 단위로 표시한다."""
    amount_krw = (
        round(amount_krw / WON_PER_MANWON) * WON_PER_MANWON
    )
    eok, remainder = divmod(amount_krw, 100_000_000)
    manwon = remainder // WON_PER_MANWON
    parts = []
    if eok:
        parts.append(f"{eok:,}억")
    if manwon:
        parts.append(f"{manwon:,}만")
    return " ".join(parts) + "원" if parts else "0원"


def us_tax_krw(principal_krw: float) -> float:
    """미국계좌의 양도소득세를 계산한다."""
    profit_krw = principal_krw * RETURN_RATE
    return max(profit_krw - US_DEDUCTION_KRW, 0.0) * US_TAX_RATE


def isa_tax_krw(principal_krw: float) -> float:
    """ISA의 분리과세액을 계산한다."""
    profit_krw = principal_krw * RETURN_RATE
    return max(profit_krw - ISA_EXEMPTION_KRW, 0.0) * ISA_TAX_RATE


def isa_effective_rate_tax_krw(principal_krw: float) -> float:
    """실효세율 비교용 ISA 세금을 수익 전액에 적용한다."""
    return principal_krw * RETURN_RATE * ISA_TAX_RATE


def us_effective_rate_tax_krw(principal_krw: float) -> float:
    """실효세율 비교용 미국계좌 세금을 투자금 구간에 따라 계산한다."""
    taxable_principal_krw = max(
        principal_krw - US_EFFECTIVE_RATE_EXEMPT_PRINCIPAL_KRW, 0.0
    )
    return taxable_principal_krw * RETURN_RATE * US_TAX_RATE


def effective_rate(tax_krw: float, principal_krw: float) -> float:
    """투자수익 대비 납부세금의 비율을 반환한다."""
    return tax_krw / (principal_krw * RETURN_RATE)


def format_rate(rate: float) -> str:
    """세율을 소수점 둘째 자리까지 표시한다."""
    return f"{rate:.1%}"


def lowest_tax_account(principal_krw: float) -> str:
    """납입·과세 기준 안에서 세금이 가장 적은 계좌를 반환한다."""
    profit_krw = principal_krw * RETURN_RATE
    candidates = [
        ("미국계좌", us_effective_rate_tax_krw(principal_krw)),
    ]
    if principal_krw <= GENERAL_MAX_KRW:
        candidates.append(("국내 일반계좌", profit_krw * GENERAL_TAX_RATE))
    if principal_krw <= ISA_MAX_KRW:
        candidates.append(("ISA", isa_effective_rate_tax_krw(principal_krw)))
    candidates.append(("연금저축", profit_krw * PENSION_TAX_RATE))
    minimum_tax_krw = min(tax_krw for _, tax_krw in candidates)
    return ", ".join(
        account
        for account, tax_krw in candidates
        if abs(tax_krw - minimum_tax_krw) < 1
    )


CHARACTERISTIC_ROWS = (
    ("투자상품", "미국 주식·ETF", "국내상장 해외 ETF", "국내상장 해외 ETF", "국내상장 해외 ETF"),
    ("납입 한도", "없음", "없음", "연 2,000만원¹", "연 1,800만원²"),
    ("과세 시점", "매도 시", "매도 시", "만기·해지 시", "인출 시"),
    ("주요 세율", "22.0%", "15.4%", "9.9%", "중도인출 시 16.5%"),
    ("기본공제·비과세", "연 250만원 공제", "없음", "200만원³ 비과세", "없음"),
    ("종합과세 여부", "배당 가능 / 양도차익 제외", "가능", "제외", "제외"),
    ("자금 인출", "자유", "자유", "원금 중도인출 가능", "중도인출 가능"),
)
CHARACTERISTIC_FOOTNOTES = (
    "¹ 실제 ISA는 연 2,000만원, 총 1억원까지 납입 가능하며 미사용 한도는 이월 가능",
    "² 실제 연금계좌 납입한도는 연금저축과 IRP 등을 합산하여 연 1,800만원",
    "³ ISA 일반형 200만원, 서민형·농어민형 400만원 비과세",
)


EFFECTIVE_RATE_AMOUNTS_KRW = (
    25_000_000,
    100_000_000,
    200_000_000,
    300_000_000,
)
EFFECTIVE_RATE_ROWS = tuple(
    (
        format_korean_currency(principal_krw),
        (
            "0.0%"
            if principal_krw <= US_EFFECTIVE_RATE_EXEMPT_PRINCIPAL_KRW
            else "22%"
        ),
        (
            format_rate(GENERAL_TAX_RATE)
            if principal_krw <= GENERAL_MAX_KRW
            else f"{GENERAL_TAX_RATE:.1%} 이상"
        ),
        (
            "9.9%"
            if principal_krw <= ISA_MAX_KRW
            else "한도 초과"
        ),
        format_rate(PENSION_TAX_RATE),
        lowest_tax_account(principal_krw),
    )
    for principal_krw in EFFECTIVE_RATE_AMOUNTS_KRW
)

PRIORITY_ROWS = (
    (
        "1",
        "미국계좌 공제분",
        format_korean_currency(US_EFFECTIVE_RATE_EXEMPT_PRINCIPAL_KRW),
        "연 250만원 기본공제 활용",
    ),
    ("2", "ISA", format_korean_currency(ISA_MAX_KRW), "5년간 최대 납입한도"),
    ("3", "연금저축", "가능한 한도까지", "연 1,800만원"),
    (
        "4",
        "국내 일반계좌",
        format_korean_currency(GENERAL_MAX_KRW),
        "종합과세 회피 기준¹",
    ),
    ("5", "미국계좌 과세분", "남은금액", ""),
)

def table_html(
    title: str,
    caption: str,
    columns: tuple[str, ...],
    rows: tuple[tuple[str, ...], ...],
    alignments: tuple[str, ...],
) -> str:
    """제목과 설명을 포함한 비교표 HTML을 만든다."""
    def row_class(label: str) -> str:
        if label == f"{MAX_WITHDRAWAL_YEARS}년":
            return "highlight-row"
        if label == "세전 대비 감소액":
            return "summary-row summary-start"
        if label == "실효세율":
            return "summary-row"
        return ""

    header = "".join(f"<th>{column}</th>" for column in columns)
    body = "".join(
        f'<tr class="{row_class(row[0])}">'
        + "".join(
            f'<td class="{alignment}">{value}</td>'
            for value, alignment in zip(row, alignments)
        )
        + "</tr>"
        for row in rows
    )
    caption_html = (
        f'<p class="result-caption">{caption}</p>' if caption else ""
    )
    return f"""
    <h2 class="result-title">{title}</h2>
    {caption_html}
    <div class="table-wrap">
      <table class="result-table">
        <thead><tr>{header}</tr></thead><tbody>{body}</tbody>
      </table>
    </div>
    """


def page_html(section: str) -> str:
    """요청한 표 구역을 공통 스타일의 HTML로 만든다."""
    characteristic_table = table_html(
        "투자계좌별 주요 특징 비교",
        "실제 계좌 제도의 일반적인 조건이며, 이번 계산의 별도 가정은 위 내용을 따릅니다.",
        ("구분", "미국계좌", "국내 일반계좌", "ISA", "연금저축"),
        CHARACTERISTIC_ROWS,
        ("identifier",) * 5,
    )
    footnotes = "".join(
        f"<p>{footnote}</p>" for footnote in CHARACTERISTIC_FOOTNOTES
    )
    effective_rate_table = table_html(
        "투자금액별 절세 계좌 선택 기준",
        f"연 {연수익률_퍼센트:g}%의 투자 수익률을 가정하였다.",
        ("투자금액", "미국계좌", "국내 일반계좌", "ISA", "연금저축", "최저세율 계좌"),
        EFFECTIVE_RATE_ROWS,
        ("identifier", "number", "number", "number", "number", "identifier best-account"),
    )
    limit_footnote = (
        '<div class="table-footnotes">'
        '<p>※ 미국계좌: 연 10% 수익률 기준, 투자금 2,500만 원까지 기본공제(0%) 적용 후 초과 수익에 22% 적용</p>'
        '<p>※ ISA계좌: 비과세 혜택은 제외하고, 발생 수익 전액에 9.9% 분리과세 보수적 적용</p>'
        '<p>※ 국내 일반계좌: 다른 금융소득이 없다고 가정하며, 연간 수익 2,000만 원 초과 시 종합과세 대상 가능</p>'
        '</div>'
    )
    priority_table = table_html(
        "절세 계좌 배분 전략",
        f"연 수익률 {RETURN_RATE:.1%}를 가정한다.",
        ("순위", "계좌", "투자금", "비고"),
        PRIORITY_ROWS,
        ("identifier", "identifier", "number", "feature"),
    )
    priority_footnote = (
        '<div class="table-footnotes">'
        '<p>¹ 국내 일반계좌는 다른 이자·배당소득이 없으며, 연 2,000만원 한도로 이익을 실현한 후 재투자한다고 가정한다.</p>'
        '</div>'
    )
    if section == "characteristics":
        content = (
            characteristic_table
            + f'<div class="table-footnotes">{footnotes}</div>'
        )
    elif section == "calculations":
        content = effective_rate_table + limit_footnote
    elif section == "priority":
        content = priority_table + priority_footnote
    else:
        raise ValueError(f"알 수 없는 표 구역입니다: {section}")
    return f"""
    <style>
    .result-title {{max-width:700px; margin:30px 0 8px; color:#0f172a;
      font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
      font-size:20px; font-weight:700; text-align:left;}}
    .result-caption {{max-width:700px; margin:0 0 10px; color:#64748b;
      font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
      font-size:13px;}}
    .table-wrap {{max-width:700px; margin:12px 0 28px; overflow-x:auto;
      border:1px solid #f0f2f5; border-radius:12px;}}
    .result-table {{width:100%; border-collapse:separate; border-spacing:0;
      font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
      font-size:13px; color:#1e293b; font-variant-numeric:tabular-nums;}}
    .result-table th {{padding:8px 12px; background:#2b4a75;
      border-bottom:2px solid #7fb3d5; color:white; font-weight:700;
      text-align:center; white-space:nowrap;}}
    .result-table td {{padding:7px 12px; border-bottom:1px solid #f0f2f5;
      background:white; text-align:center; white-space:nowrap;}}
    .result-table .identifier {{text-align:center;}}
    .result-table .number {{text-align:right;}}
    .result-table .feature {{text-align:left;}}
    .result-table .best-account {{font-weight:700;}}
    .result-table tbody tr:nth-child(even) td {{background:#fafbfc;}}
    .result-table tbody tr:hover td {{background:#eff6ff;}}
    .result-table tbody tr:last-child td {{border-bottom:0;}}
    .result-table tbody .highlight-row td {{font-weight:700;}}
    .result-table tbody .summary-row td {{background:#e8edf3;
      font-weight:600;}}
    .result-table tbody .summary-start td {{border-top:2px solid #94a3b8;}}
    .table-footnotes {{max-width:700px; margin:-18px 0 28px; color:#64748b;
      font-family:Pretendard,-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
      font-size:13px;}}
    .table-footnotes p {{margin:4px 0;}}
    </style>
    {content}
    """


def validate_inputs() -> None:
    """그래프 계산에 사용하는 입력값을 검증한다."""
    if not 0 < INITIAL_INVESTMENT_KRW <= ISA_MAX_KRW:
        raise ValueError("초기 투자금은 0원 초과 1억원 이하로 입력하세요.")
    if RETURN_RATE <= 0:
        raise ValueError("연 수익률은 0%보다 크게 입력하세요.")
    if not 1 <= MAX_WITHDRAWAL_YEARS <= 10:
        raise ValueError("최대 인출 시점은 1년 이상 10년 이하로 입력하세요.")
    if GENERAL_SALE_MODE not in (
        GENERAL_ANNUAL_SALE_MODE, GENERAL_THRESHOLD_SALE_MODE
    ):
        raise ValueError("국내 일반계좌 매도 방식을 목록에서 선택하세요.")


def calculate_after_tax_withdrawals(
    initial_investment_krw: float = INITIAL_INVESTMENT_KRW,
) -> dict[str, list]:
    """종료연도별 계좌의 세후 전액 인출액을 계산한다."""
    results: dict[str, list] = {
        "미국계좌": [],
        "국내 일반계좌": [],
        "ISA": [],
        "연금저축": [],
    }
    general_balance_krw = initial_investment_krw
    general_basis_krw = initial_investment_krw

    for year in range(1, MAX_WITHDRAWAL_YEARS + 1):
        general_balance_krw *= 1 + RETURN_RATE
        general_unrealized_gain_krw = max(
            general_balance_krw - general_basis_krw, 0.0
        )
        if GENERAL_SALE_MODE == GENERAL_ANNUAL_SALE_MODE:
            general_realized_gain_krw = general_unrealized_gain_krw
        else:
            general_realized_gain_krw = (
                FINANCIAL_INCOME_LIMIT_KRW
                if general_unrealized_gain_krw > FINANCIAL_INCOME_LIMIT_KRW
                else 0.0
            )
        general_tax_krw = general_realized_gain_krw * GENERAL_TAX_RATE
        general_balance_krw -= general_tax_krw
        general_basis_krw += general_realized_gain_krw - general_tax_krw
        remaining_gain_krw = max(
            general_balance_krw - general_basis_krw, 0.0
        )
        hypothetical_liquidation_tax_krw = (
            remaining_gain_krw * GENERAL_TAX_RATE
        )
        results["국내 일반계좌"].append(
            general_balance_krw - hypothetical_liquidation_tax_krw
        )

        gross_balance_krw = initial_investment_krw * (1 + RETURN_RATE) ** year
        cumulative_profit_krw = gross_balance_krw - initial_investment_krw
        us_final_tax_krw = cumulative_profit_krw * US_TAX_RATE
        isa_tax_krw = (
            max(cumulative_profit_krw - ISA_EXEMPTION_KRW, 0.0)
            * ISA_TAX_RATE
        )
        pension_tax_krw = cumulative_profit_krw * PENSION_TAX_RATE
        results["미국계좌"].append(gross_balance_krw - us_final_tax_krw)
        results["ISA"].append(gross_balance_krw - isa_tax_krw)
        results["연금저축"].append(
            gross_balance_krw - pension_tax_krw
        )

    return results


def select_font() -> str:
    """설치된 한글 글꼴 중 우선순위가 가장 높은 글꼴을 선택한다."""
    from matplotlib import font_manager

    candidates = (
        "Pretendard",
        "Apple SD Gothic Neo",
        "Noto Sans CJK KR",
        "Malgun Gothic",
    )
    installed = {font.name for font in font_manager.fontManager.ttflist}
    return next((font for font in candidates if font in installed), "DejaVu Sans")


def create_withdrawal_chart(
    results: dict[str, list],
):
    """종료연도별 세후 전액 인출액 선그래프를 만든다."""
    import matplotlib.pyplot as plt

    font_name = select_font()
    plt.rcParams["font.family"] = font_name
    plt.rcParams["axes.unicode_minus"] = False
    colors = {
        "미국계좌": "#2F7DD3",
        "국내 일반계좌": "#F06432",
        "ISA": "#1FAE7A",
        "연금저축": "#F2A000",
    }
    years = list(range(1, MAX_WITHDRAWAL_YEARS + 1))
    fig, ax = plt.subplots(figsize=(10, 6.5))
    fig.patch.set_facecolor("#FFFFFF")
    ax.set_facecolor("#FFFFFF")

    for account, balances in results.items():
        values_eok = [
            balance / WON_PER_EOK if balance is not None else None
            for balance in balances
        ]
        ax.plot(
            years,
            values_eok,
            color=colors[account],
            linewidth=3.2,
            linestyle="-",
            label=account,
        )

    initial_eok = INITIAL_INVESTMENT_KRW / WON_PER_EOK
    ax.axhline(initial_eok, color="#F59E0B", linewidth=1.4, linestyle=":")
    ax.text(
        1,
        initial_eok,
        f"시작 투자금 {format_korean_currency(INITIAL_INVESTMENT_KRW)}",
        color="#F59E0B",
        fontsize=14,
        va="bottom",
    )

    label_offsets = {"미국계좌": -2, "ISA": 4, "연금저축": 14}
    for account in ("미국계좌", "ISA", "연금저축"):
        final_balance_krw = results[account][-1]
        ax.annotate(
            format_korean_currency(final_balance_krw),
            xy=(MAX_WITHDRAWAL_YEARS, final_balance_krw / WON_PER_EOK),
            xytext=(8, label_offsets[account]),
            textcoords="offset points",
            color=colors[account],
            fontsize=14,
            fontweight="bold",
            va="center",
        )

    final_general_krw = results["국내 일반계좌"][-1]
    ax.annotate(
        format_korean_currency(final_general_krw),
        xy=(MAX_WITHDRAWAL_YEARS, final_general_krw / WON_PER_EOK),
        xytext=(8, -16),
        textcoords="offset points",
        color=colors["국내 일반계좌"],
        fontsize=14,
        fontweight="bold",
        va="center",
    )

    ax.set_xlim(1, MAX_WITHDRAWAL_YEARS + 0.8)
    ax.set_xticks(years)
    ax.set_xlabel("전액 인출 시점(년)", fontsize=15, color="#777777")
    ax.set_ylabel("세후 인출액(억원)", fontsize=15, color="#777777")
    ax.grid(axis="y", color="#DEDCD6", linewidth=0.9)
    ax.grid(axis="x", visible=False)
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#DEDCD6")
    ax.tick_params(axis="both", colors="#777777", labelsize=15, length=0)
    ax.legend(
        loc="lower left",
        bbox_to_anchor=(0, 1.01),
        ncol=2,
        frameon=False,
        prop={"size": 15},
    )
    fig.text(0.10, 0.97, "대도시 연구실", fontsize=13, color="#777777")
    fig.text(
        0.10,
        0.925,
        "전액 인출 시점별 세후 금액 | 계좌 유형 비교",
        fontsize=21,
        fontweight="bold",
        color="#0B0B0B",
    )
    fig.text(
        0.10,
        0.88,
        f"초기 투자금 {format_korean_currency(INITIAL_INVESTMENT_KRW)} · 연 수익률 {RETURN_RATE:.1%} · 최대 {MAX_WITHDRAWAL_YEARS}년",
        fontsize=16,
        color="#666666",
    )
    fig.text(
        0.10,
        0.025,
        "※ 분배금·수수료·환율 제외 · 국내 일반계좌 금융소득 종합과세 추가세금 미반영",
        fontsize=13,
        color="#777777",
    )
    fig.subplots_adjust(left=0.10, right=0.86, top=0.76, bottom=0.16)
    return fig


def save_chart(fig):
    """그래프를 PNG로 저장하고 경로를 반환한다."""
    import sys
    from pathlib import Path

    output_dir = Path("/content/output") if "google.colab" in sys.modules else Path("output")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "qqq_account_after_tax_withdrawal.png"
    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
        facecolor="#FFFFFF",
    )
    return output_path


def withdrawal_yearly_table_html(
    results: dict[str, list],
    initial_investment_krw: float,
) -> str:
    """연도별 계좌의 세후 전액 인출액 비교표를 만든다."""
    from itertools import combinations

    ranking_order = ("ISA", "국내 일반계좌", "연금저축", "미국계좌")

    def compare(account_a: str, account_b: str, index: int) -> int:
        difference_krw = results[account_a][index] - results[account_b][index]
        if abs(difference_krw) < 1:
            return 0
        return 1 if difference_krw > 0 else -1

    rows = []
    reversal_records = []
    for index in range(MAX_WITHDRAWAL_YEARS):
        year = index + 1
        if index == 0:
            note = ""
        else:
            changes = []
            for account_a, account_b in combinations(ranking_order, 2):
                previous = compare(account_a, account_b, index - 1)
                current = compare(account_a, account_b, index)
                if current == previous or current == 0:
                    continue
                if previous == 0 or previous * current < 0:
                    winner = account_a if current > 0 else account_b
                    loser = account_b if current > 0 else account_a
                    subject = "연금저축이" if winner == "연금저축" else f"{winner}가"
                    changes.append(f"{subject} {loser} 역전")
            note = " · ".join(changes)
            if note:
                reversal_records.append(f"{year}년 차 {note}")
        pretax_balance_krw = initial_investment_krw * (1 + RETURN_RATE) ** year
        rows.append(
            (
                f"{year}년",
                format_korean_currency(pretax_balance_krw),
                format_korean_currency(results["미국계좌"][index]),
                format_korean_currency(results["국내 일반계좌"][index]),
                format_korean_currency(results["ISA"][index]),
                format_korean_currency(results["연금저축"][index]),
            )
        )
    pretax_final_krw = initial_investment_krw * (
        1 + RETURN_RATE
    ) ** MAX_WITHDRAWAL_YEARS
    final_balances = {
        "세전": pretax_final_krw,
        **{account: values[-1] for account, values in results.items()},
    }
    decrease_by_account = {
        account: max(pretax_final_krw - balance_krw, 0.0)
        for account, balance_krw in final_balances.items()
    }
    rows.append((
        "세전 대비 감소액",
        *(
            format_korean_currency(decrease_by_account[account])
            for account in ("세전", "미국계좌", "국내 일반계좌", "ISA", "연금저축")
        ),
    ))
    pretax_profit_krw = pretax_final_krw - initial_investment_krw
    rows.append((
        "실효세율",
        *(
            format_rate(decrease_by_account[account] / pretax_profit_krw)
            for account in ("세전", "미국계좌", "국내 일반계좌", "ISA", "연금저축")
        ),
    ))
    table = table_html(
        "전액 인출 시점별 세후 금액",
        f"초기 투자금 {format_korean_currency(initial_investment_krw)} · 연 수익률 {RETURN_RATE:.1%} · 세금 차감 후 전액 인출액",
        ("연도", "세전", "미국계좌", "국내 일반계좌", "ISA", "연금저축"),
        tuple(rows),
        ("identifier", "number", "number", "number", "number", "number"),
    )
    if GENERAL_SALE_MODE == GENERAL_ANNUAL_SALE_MODE:
        general_strategy_footnote = (
            '<p>※ 국내 일반계좌는 금융소득종합과세를 방지하기 위해 매년 전량 매도하고, 세금 차감 후 전액 재투자한다고 가정한다.</p>'
        )
    else:
        general_strategy_footnote = (
            '<p>※ 국내 일반계좌는 과세 대상 이익이 2,000만원을 초과할 때 이익 2,000만원만 실현한 후 재매수한다.</p>'
        )
    reversal_footnote = (
        f'<p>※ 역전 기록: {" · ".join(reversal_records)}</p>'
        if reversal_records
        else ""
    )
    footnote = (
        '<div class="table-footnotes">'
        + reversal_footnote
        + '<p>※ 절세 계좌 배분 전략의 3순위를 찾기 위한 실험이므로 미국계좌의 250만원 공제는 이미 사용되었다고 가정한다.</p>'
        + '<p>※ 실효세율은 세전 투자이익 대비 세전 대비 감소액의 비율로 계산하였다.</p>'
        + general_strategy_footnote
        + '</div>'
    )
    return table + footnote


def main() -> None:
    """계좌별 세금 표와 보유기간별 세후 인출액을 표시한다."""
    from IPython.display import HTML, Markdown, display

    validate_inputs()
    display(HTML(page_html("characteristics")))
    display(Markdown(ASSUMPTIONS_MD))
    display(HTML(page_html("calculations")))
    display(Markdown(WITHDRAWAL_ASSUMPTIONS_MD))
    results = calculate_after_tax_withdrawals(INITIAL_INVESTMENT_KRW)
    display(HTML(withdrawal_yearly_table_html(results, INITIAL_INVESTMENT_KRW)))
    display(HTML(page_html("priority")))


main()
